In [ ]:
# Week 1 Day 2 Homework

### I made a translator that summarizes websites and translates the summary into Middle Engilsh.  It turns
### out that small open-source language models don't know Middle English, so I provided some vocabulary in the system prompt.
### Check out the results below.  I used gemma4:e2b for this project.

### Import the necessary modules.

In [ ]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests

### Make sure Ollama is on your machine.

In [5]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download gemma4:e2b from Ollama.
### Note: this took over 3 minutes on my laptop.

In [6]:
!ollama pull gemma4:e2b

]11;?\pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏ 837 KB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏ 5.0 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏ 7.1 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  14 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  19 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  22 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  25 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  32 MB/7.2 GB      

In [7]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

### Set up the system prompt.

In [37]:
system_prompt = """
You are a professor of Middle English who analyzes the contents of a website,
summarizes them, and then translates that summary from Modern English into Middle English, 
ignoring text that might be 
navigation-related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
Do not return the Modern English summary.  Do not mention that you are summarizing or translating anything.
Just return the translation itself.
Please use the following as reference:

Al be that - Although
Anon - At once; at another time
Bet - Better
Can - Know; be able
Cas - Happening now; chance
Coy - Quiet
Echo - Each one
Everich - Every; every one
Forthy - Therefore
Han - Have
Ich - I
Kan - Know; know how to; can
Lite - Little
Moot - May; must; ought to; so
Nat - Not
Noon - None; no
Nyce: foolish
Pryme - 9 A.M.
Rede - Advise; interpret; read
Shaltow - You shall
Thilke - This; that; at that
Tho - Those; then
Unnethe - Scarcely
Ynogh - Enough 
Array - Arrangement or condition
Bane - Destruction
Boote - Remedy
Certeyn - Certain
Deel - Part or bit
Devyse - Trick or device
Fetis - Well made
Gentle - Noble
Hende - Handy or courteous
Leef - Dear
Mete - Food
Ny - Near
Paas - Pace or slow walk
Routh - Pity
Siker - Trusty
Verray - True

Check out this “The Knight’s Tale” excerpt from The Canterbury Tales:

    Whilom, as olde stories tellen us,

    Ther was a duc that highte Theseus;

    Of Atthenes he was lord and governour,

In Modern English, the three lines could be rewritten like this:

    Once, as old histories tell us,

    There was a duke who was called Theseus;

    He was lord and governor of Athens,

    
    Apprise - Inform people
Burnish - Polish
Courtly - Refined appearance
Din - Harsh noise
Espy - Catch the eye
Fetter - Shackles; restrained in shackles
Gilded - Made or covered in gold
Hauberk - Chainmail covering the neck and shoulders
Implore - Beg
Liege - Lord
Mirth - Merriment
Popinjay - Parrot
Revel - Unrestrained merriment
Stout - Dependable or courage
Unsullied - Clean or spotless
Verily - In truth
Wayward - Undisciplined; resist guidance

Check out this excerpt from part one of Sir Gawain and the Green Knight in its original Middle English:

    And neuenes hit his aune nome, as hit now hat;

    Tirius to Tuskan and teldes bigynnes,

    Langaberde in Lumbardie lyftes vp homes,

And here are the same lines in a modern English translation:

    And names it with his own name, which it now has;

    Tirius turns to Tuscany and founds dwellings;

    Longobard raises home in Lombardy;

"""

### Set up the user prompt.

In [38]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website, translating it from Modern English to Middle English.
Especially focus on any sports-related news or announcements.

"""

### Define a function for sending messages to the llm.

In [39]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

### Define a function to call the gemma4 API and have it summarize a website.

In [40]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "gemma4:e2b",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

### Add a function to display the output nicely in Markdown.

In [41]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

### Let's try some websites!

In [42]:
display_summary('https://www.bbc.com')

Whilom, as olde cinstories tellen us, þis is the home of the BBC where many matters are writen. In the news there are talkes of the war in Ukraine and Russia; the king of Ukraine did propose face-to-face talkes to Putin in an open letter, for that only direct engagement betwixt the two lands could end the war, with the King of England oft weorce on Iran. Also is writen that the ex-wife of the nephew of the ruler of Dubai is in custody, which is a matter of property arrangement.

In [43]:
display_summary('https://www.rolandgarros.com/en-us/')

Þis is <0xC7><0xA3>n sit for the Roland-Garros Eorðe, which is a tournament.

For the year of Twenty-and-Six, the strete was set from the eighteenth of May unto the seventh of June.

There are noteworth þings and writes of the torne, such as the wrap that was done on the fourth of June.

Featurenes do tell of Andreeva who is focused in her endwist, and of Maja, who is a tennis freak.

A match report telleth that Unheralded Chwalinska won the maiden Slam final.